In [ ]:
import os
from glob import glob
import pandas as pd
import nibabel as nib
from nilearn import plotting
from scipy.ndimage import center_of_mass
import numpy as np
import sys
import math

repository_path = "/Users/user/Downloads/longitudinal_sift2/"
functions_path  = os.path.join(repository_path, "code", "paper_figures", "functions")
data_dir = os.path.join(repository_path, "data", "in_vivo")
dependencies_dir = os.path.join(repository_path, "code", "paper_figures", "dependencies")
sys.path.append(os.path.abspath(functions_path))

from define_study import define_study
from prepare_connectomes import prepare_connectomes

### DEFINE DATASET

In [ ]:
dataset = "Templobe_Surgery" #"HCP_Scan_Rescan" #Developing_Children #Templobe_Surgery
ntcks = "10M"
results_dir = os.path.join(data_dir, dataset)

### LOAD DATA

In [ ]:
# load labels
labels = pd.read_csv(os.path.join(dependencies_dir,"fs_default.txt"), comment='#', delim_whitespace=True, header=None,names=['ID', 'Code', 'Description', 'R', 'G', 'B', 'A'])["Description"][1:86]

# Define the parcellation image in MNI
parcellation_img = nib.load(f"{dependencies_dir}/Desikan-Killiany_left_right_in_MNI.nii.gz")

# Select the full connectomes without the surg_defect label (index 86)
subset_idx = (1,85)

Define Paths

In [ ]:
# Longitudinal Difference  
fbc_differences_none_path = f"{results_dir}/effects/{ntcks}/sift2_none/fbc_differences"
fbc_differences_cross_path = f"{results_dir}/effects/{ntcks}/sift2_cross/fbc_differences"
fbc_differences_sym_path = f"{results_dir}/effects/{ntcks}/sift2_symmetric/fbc_differences"
fbc_differences_diff_path = f"{results_dir}/effects/{ntcks}/sift2_differential/fbc_differences"


Pipeline 1

In [ ]:
# load connectome files
fbc_differences_none_files = glob(os.path.join(fbc_differences_none_path, "sub*.csv"))

# load connectome dicts
fbc_differences_none_dict = prepare_connectomes(fbc_differences_none_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_none = [data for data in fbc_differences_none_dict.values()]

# calculate the mean matrices
fbc_differences_mean_none = pd.DataFrame(np.mean(np.array(fbc_differences_none), axis=0))

# calculate the stdv matrices
fbc_differences_std_none = pd.DataFrame(np.std(np.array(fbc_differences_none), axis=0))

Pipeline 2

In [ ]:
## LOAD cross CONNECTOME MATRICES
# define path
# load connectome files
fbc_differences_cross_files = glob(os.path.join(fbc_differences_cross_path, "sub*.csv"))

# load connectome dicts
fbc_differences_cross_dict = prepare_connectomes(fbc_differences_cross_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_cross = [data for data in fbc_differences_cross_dict.values()]

# calculate the mean matrices
fbc_differences_mean_cross = pd.DataFrame(np.mean(np.array(fbc_differences_cross), axis=0))

# calculate the stdv matrices
fbc_differences_std_cross = pd.DataFrame(np.std(np.array(fbc_differences_cross), axis=0))

Pipeline 3

In [ ]:
## LOAD UNB CONNECTOME MATRICES
# define path

# load connectome files
fbc_differences_sym_files = glob(os.path.join(fbc_differences_sym_path, "sub*.csv"))

# load connectome dicts
fbc_differences_sym_dict = prepare_connectomes(fbc_differences_sym_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_sym = [data for data in fbc_differences_sym_dict.values()]

# calculate the mean matrices
fbc_differences_mean_sym = pd.DataFrame(np.mean(np.array(fbc_differences_sym), axis=0))

# calculate the stdv matrices
fbc_differences_std_sym = pd.DataFrame(np.std(np.array(fbc_differences_sym), axis=0))

In [ ]:
## LOAD diff CONNECTOME MATRICES
# define path

# load connectome files
fbc_differences_diff_files = glob(os.path.join(fbc_differences_diff_path, "sub*.csv"))

# load connectome dicts
fbc_differences_diff_dict = prepare_connectomes(fbc_differences_diff_files, subset_idx=subset_idx, start_idx_1=True)

# Extract the connectome matrices from the filtered dictionary
fbc_differences_diff = [data for data in fbc_differences_diff_dict.values()]

# calculate the mean matrices
fbc_differences_mean_diff = pd.DataFrame(np.mean(np.array(fbc_differences_diff), axis=0))

# calculate the stdv matrices
fbc_differences_std_diff = pd.DataFrame(np.std(np.array(fbc_differences_diff), axis=0))

PREPARE VISUALISATION

In [ ]:
# Load the parcellation image
parcellation_data = parcellation_img.get_fdata()

# Extract unique labels (excluding the background label 0)
labels = np.unique(parcellation_data)
labels = labels[labels != 0]

# Compute the center of mass for each label
label_coords = {}
for label in labels:
    coords = center_of_mass(parcellation_data == label)
    # Convert voxel coordinates to MNI coordinates
    mni_coords = nib.affines.apply_affine(parcellation_img.affine, coords)
    label_coords[label] = mni_coords
    
# Extract the coordinates and ensure they are in the correct order
coords = [label_coords[label] for label in sorted(label_coords.keys())][:86]

### PLOT EDGE-WISE VARIANCE

In [ ]:
# Define Visualisation Params
edge_threshold = None
edge_vmin = 0
edge_vmax = 2.5
edge_cmap = "hot"
node_color = "lightgrey"

counter=1

for matrix, title in [(fbc_differences_std_none,"Pipeline 1"), (fbc_differences_std_cross,"Pipeline 2"),(fbc_differences_std_sym,"Pipeline 3"),(fbc_differences_std_diff,"Pipeline 4")]:
    # Plot the connectome
    plotting.plot_connectome(matrix, coords, edge_cmap=edge_cmap, edge_threshold = edge_threshold, edge_vmin = edge_vmin, edge_vmax = edge_vmax, node_color=node_color, title=title, radiological = True, colorbar=False, display_mode='ortho')

    counter += 1

### TESTING FOR STATISTICAL DIFFERENCES BETWEEN PIPELINES

In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --------------------------------------------------------
# 1. Convert dfs to dicts
# --------------------------------------------------------
dfs = {
    "none": fbc_differences_std_none,
    "cross": fbc_differences_std_cross,
    "sym": fbc_differences_std_sym,
    "diff": fbc_differences_std_diff,
}

# Extract upper triangle 
def extract_upper_incl_diag(df):
    arr = df.values
    iu = np.triu_indices_from(arr, k=0)  # include diagonal
    return arr[iu]

upper_values = {name: extract_upper_incl_diag(df) for name, df in dfs.items()}

# Long-format DataFrame for ANOVA
anova_df = pd.DataFrame({
    "Error": np.concatenate([vals for vals in upper_values.values()]),
    "Category": np.concatenate([[name] * len(vals) for name, vals in upper_values.items()])
})

# p-value classification
def p_threshold(p):
    if p < 0.001:
        return "< 0.001"
    elif p < 0.01:
        return "< 0.01"
    elif p < 0.05:
        return "< 0.05"
    else:
        return "n.s."

# One-way ANOVA
groups = [anova_df.loc[anova_df.Category == c, "Error"] for c in dfs.keys()]
F, p_anova = f_oneway(*groups)

print(f"\nANOVA F = {F:.3f}, p = {p_anova:.3g}  ({p_threshold(p_anova)})\n")


# Tukey HSD post-hoc test
tukey = pairwise_tukeyhsd(
    endog=anova_df["Error"],
    groups=anova_df["Category"],
    alpha=0.05
)

### PRINT RESULTS ### 

print("Tukey HSD results:\n")
print(tukey.summary())

# Print pairwise comparisons with thresholds
print("\nPairwise comparisons with thresholds:\n")

# Tukey results are inside a table-like structure
for comp in tukey._results_table.data[1:]:
    g1, g2, meandiff, p_adj, lower, upper, reject = comp
    print(f"{g1} vs {g2}: p = {float(p_adj):.3g}  ({p_threshold(float(p_adj))})")